# 経済データ分析Ⅱ　第5回
# 不均一分散と頑健な標準誤差

第2回では47都道府県の持ち家の床面積を人口密度の対数に回帰した（以下、住宅データ）。残差の図では、人口密度の低い県ほど残差が上下に大きく広がっていた。誤差の分散が観測によって違うことを**不均一分散**といい、残差の広がりの違いはその手掛かりになる。第2回の表に載せた傾きの標準誤差 1.85 は、誤差の分散がどの県でも同じという仮定（第3回の仮定5）の下で計算した値である。

この Notebook では、まず第4回で使った式の書き換えを1つ補足する。そのあとで、不均一分散があると通常の標準誤差がどれだけずれるのか、そのずれをどう直すのかを、第4回で導いた係数の分散の式をもとに考える。前半では誤差の分散が所得とともに大きくなる人工データを作り、通常の標準誤差のずれを調べる。後半で住宅データに戻り、不均一分散を許す標準誤差を計算する。

## Notebook の使い方

コードセルを上から順に選び、`Shift + Enter` で実行します。途中のセルから実行して「変数が見つからない」というエラー（`NameError`）が出たら、先頭のセルから実行し直してください。

JupyterLite では **Python (Pyodide)**、ローカルの Jupyter では Python カーネルを使います。データは Notebook の中で作るか Notebook に収録しているので、CSV ファイルの準備は要りません。

## 1. 準備

次のセルで、今回使うライブラリを読み込む。`japanize_matplotlib_jlite` は、図の文字を日本語で表示するためのパッケージで、経済データ分析Ⅰでも使った。ブラウザでは `piplite.install()` でこのパッケージを入れる。`if sys.platform == 'emscripten':` から始まる部分は、ブラウザ（JupyterLite）で実行しているときに必要なライブラリを用意する処理である。`np`、`pd`、`plt`、`sm` は第2回・第3回と同じ略称で、それぞれ NumPy、pandas、Matplotlib、statsmodels を指す。

In [ ]:
import sys
if sys.platform == 'emscripten':
    import pyodide_js
    await pyodide_js.loadPackage(['numpy', 'pandas', 'matplotlib', 'statsmodels'])
    import piplite
    await piplite.install('japanize-matplotlib-jlite')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib_jlite
import statsmodels.api as sm

## 2. 傾きの式の分子の書き換え

第4回の第3.1節では、OLS の傾きの式の分子 $\sum(x_{1i}-\bar x_1)(y_i-\bar y)$ を $\sum(x_{1i}-\bar x_1)y_i$ と書き換えた。$y_i$ から平均 $\bar y$ を引いても引かなくても和が変わらない理由を、式と例で確かめる。この節では所得を $x_i$、その平均を $\bar x$ と書く。

まず、1つの世帯の項を展開する。

$$(x_i-\bar x)(y_i-\bar y)=(x_i-\bar x)y_i-(x_i-\bar x)\bar y$$

これを全世帯で足す。$\bar y$ はどの世帯でも同じ数なので、和の外に出せる。

$$\sum_{i=1}^{n}(x_i-\bar x)(y_i-\bar y)=\sum_{i=1}^{n}(x_i-\bar x)y_i-\bar y\sum_{i=1}^{n}(x_i-\bar x)$$

右辺の最後の $\sum(x_i-\bar x)$ は所得の偏差の合計で、いつも0になる。平均は $\bar x=\sum x_i/n$ なので $\sum x_i=n\bar x$ であり、

$$\sum_{i=1}^{n}(x_i-\bar x)=\sum_{i=1}^{n}x_i-n\bar x=n\bar x-n\bar x=0$$

となるからである。そのため右辺の第2項は $\bar y\times0=0$ で、

$$\sum_{i=1}^{n}(x_i-\bar x)(y_i-\bar y)=\sum_{i=1}^{n}(x_i-\bar x)y_i$$

が成り立つ。

第4回の5世帯の例で確かめる。所得は20、30、40、50、60万円、消費は13、13、20、16、24万円だった。下のセルでは、世帯ごとに所得の偏差、消費の偏差、2通りの積を計算して表にする。

In [ ]:
x_ex = np.array([20, 30, 40, 50, 60])
y_ex = np.array([13, 13, 20, 16, 24])
table = pd.DataFrame({'所得 x': x_ex, '消費 y': y_ex,
                      'x の偏差': x_ex - x_ex.mean(),
                      'y の偏差': y_ex - y_ex.mean()})
table['偏差どうしの積'] = table['x の偏差'] * table['y の偏差']
table['x の偏差 × y'] = table['x の偏差'] * table['消費 y']
table

所得の平均は40万円、消費の平均は17.2万円である。1行目の世帯では、偏差どうしの積は $(-20)\times(-4.2)=84$、所得の偏差と消費の積は $(-20)\times13=-260$ で、世帯ごとの値は違う。`.sum()` で列ごとの合計を求める。

In [ ]:
print(f"x の偏差の合計：{table['x の偏差'].sum():.1f}")
print(f"偏差どうしの積の合計：{table['偏差どうしの積'].sum():.1f}")
print(f"x の偏差 × y の合計：{table['x の偏差 × y'].sum():.1f}")

所得の偏差の合計は0で、2つの積の合計はどちらも250になる。2つの列の差は、世帯ごとに $(x_i-\bar x)\bar y$ で、その合計が $\bar y\times0=0$ だからである。

第4回の第4節の欠落変数バイアスの式でも、同じ書き換えを使った。所得だけの回帰の傾きに、世帯人数 $x_{2i}$ を含む母集団の式を代入すると、世帯人数の項 $\beta_2\sum(x_{1i}-\bar x_1)x_{2i}/SST_1$ が残る。この分数は、世帯人数を所得に回帰した傾き $\sum(x_{1i}-\bar x_1)(x_{2i}-\bar x_2)/SST_1$ と同じ値になる。分子の $x_{2i}-\bar x_2$ を $x_{2i}$ に置き換えても和が変わらないのは、上と同じ理由である。

## 3. 所得とともにばらつきが大きくなる消費

第3回と同じく、総務省の家計調査を参考にして200世帯の人工データを作る。今回は説明変数を所得だけにして、誤差の作り方を第3回から変える。

まず乱数を作る仕組みを用意する。`np.random.default_rng(202610)` の括弧の中の数（シード）を同じにしておけば、何度実行しても同じ乱数が出る。`n` は世帯の数である。

In [ ]:
rng = np.random.default_rng(202610)
n = 200

所得は、50万円に倍率 `multiplier` を掛けて世帯ごとにばらつかせる。倍率は第3回と同じく `rng.lognormal()` で対数正規分布から選ぶ。`mean=-0.06` と `sigma=0.35` は、倍率の平均がほぼ1になるように選んだ値である。50万円は、第3回の表にある世帯人数別の平均所得（37.0〜72.2万円）の中ほどにあたる。

`print()` の中の `{income.mean():.1f}` は、所得の平均を小数第1位まで表示する書き方である。`.mean()` は平均を、`.min()` と `.max()` は最小と最大を求める。

In [ ]:
multiplier = rng.lognormal(mean=-0.06, sigma=0.35, size=n)
income = 50 * multiplier
print(f'所得の平均：{income.mean():.1f} 万円')
print(f'所得の最小：{income.min():.1f} 万円、最大：{income.max():.1f} 万円')

所得の平均は約52万円で、最も低い世帯は約19万円、最も高い世帯は約128万円である。

消費は次の式で作る。

$$消費_i=10+0.3\times 所得_i+u_i$$

所得の係数0.3は第3回と同じで、所得が1万円多い世帯ほど、消費は平均して0.3万円（3,000円）多い。

第3回と違うのは誤差 $u_i$ の作り方である。第3回ではどの世帯の誤差も標準偏差4万円だったが、今回は誤差の標準偏差を所得の1割とする。所得20万円の世帯なら2万円、所得100万円の世帯なら10万円になる。第3回で仮定5を説明したときに挙げたように、所得の高い世帯は旅行や家具・家電に使うかどうかの選択の幅が広いので、所得が同じ世帯の中でも消費のばらつきが大きくなりやすい。

誤差の平均はどの所得でも0なので、第3回の仮定1〜4は成り立つ。成り立たないのは仮定5（均一分散）である。

下のセルの `sigma` は、各世帯の誤差の標準偏差を並べたものである。`rng.normal(0, sigma)` のように標準偏差のところに `sigma` を渡すと、世帯ごとに違う標準偏差で、平均0の正規分布から200個の乱数を作る。

In [ ]:
sigma = 0.1 * income
error = rng.normal(0, sigma)
consumption = 10 + 0.3 * income + error

4つの変数を表にまとめ、先頭の5世帯を表示する。`.head()` は先頭の5行を表示し、`.round(1)` は小数第1位までに丸めて表示する。

In [ ]:
df = pd.DataFrame({'income': income, 'sigma': sigma,
                   'error': error, 'consumption': consumption})
df.head().round(1)

`income` は所得、`sigma` は誤差の標準偏差、`error` は誤差、`consumption` は消費で、単位はどれも月額・万円である。1行目の世帯は所得が53.5万円なので、誤差の標準偏差は5.4万円になっている。

## 4. 回帰直線と残差

第2回・第3回と同じ方法で、消費を所得に回帰する。`df[['income']]` で表から所得の列を取り出し、`sm.add_constant()` で切片用の列を加える。`sm.OLS(y, X).fit()` が係数を求め、結果の `fit.params` に切片（`const`）と所得の係数（`income`）が入る。

In [ ]:
X = sm.add_constant(df[['income']])
fit = sm.OLS(df['consumption'], X).fit()
print(f"切片：{fit.params['const']:.3f}")
print(f"所得の係数：{fit.params['income']:.3f}")

所得の係数の推定値は0.296で、真の値0.3に近い。

残差を縦軸、所得を横軸にした散布図を描く。残差は `fit.resid` に入っている。

- `plt.subplots()` で図を用意する。`fig` は図全体、`ax` は点を描く場所である。
- `ax.scatter(横軸, 縦軸, s=15)` で散布図を描く。`s=15` は点の大きさである。
- `ax.axhline(0, color='black')` は高さ0の位置に黒い横線を引く。
- `ax.set()` で横軸（`xlabel`）と縦軸（`ylabel`）の名前を付ける。

最後の `plt.show()` で図を表示する。

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df['income'], fit.resid, s=15)
ax.axhline(0, color='black')
ax.set(xlabel='所得（万円）', ylabel='残差（万円）')
plt.show()

所得が30万円より低い世帯の残差は、ほぼ ±5万円の範囲に収まる。所得が70万円を超える世帯には、±10万円を超える残差もいくつかある。誤差の標準偏差を所得の1割にしたので、所得が高いほど残差も上下に広がる。

## 5. 傾きの推定値と誤差

第4回の第3.1節では、単回帰の傾きの推定値を次の形に書き直した。ここでは第4回の $x_{1i}$ と $SST_1$ を $x_i$ と $SST$ と書く。

$$\hat\beta_1=\beta_1+\sum_{i=1}^{n}w_iu_i,\qquad w_i=\frac{x_i-\bar x}{SST}$$

$x_i$ は世帯 $i$ の所得、$\bar x$ は所得の平均、$SST=\sum_{i=1}^{n}(x_i-\bar x)^2$ は所得の偏差の二乗和である。推定値は、真の係数 $\beta_1$ に誤差の重み付きの和を足したものになる。重み $w_i$ は所得だけで決まり、平均より所得の高い世帯では正、低い世帯では負の値をとる。所得が平均から離れた世帯ほど、重みの絶対値は大きい。

重み $w_i$ を計算する。`x - x.mean()` は各世帯の所得から平均を引いた偏差、`**2` は2乗、`.sum()` は合計である。

In [ ]:
x = df['income']
sst = ((x - x.mean())**2).sum()
w = (x - x.mean()) / sst

人工データでは誤差 $u_i$ が分かるので、上の式を直接確かめられる。$\beta_1=0.3$ に $\sum w_iu_i$ を足した値と、OLS の所得の係数を並べる。

In [ ]:
print(f"0.3 + Σ w_i u_i：{0.3 + (w * df['error']).sum():.4f}")
print(f"OLS の所得の係数：{fit.params['income']:.4f}")

2つの値は一致する。推定値が真の値0.3から約0.004ずれたのは、200世帯の誤差の重み付きの和が0にならなかったためである。

## 6. 推定値の分散

別の200世帯を調べると、誤差 $u_i$ が変わり、推定値も変わる。所得を固定して考えると、変わるのは $\sum w_iu_i$ の部分である。誤差が互いに独立なら（第3回の仮定2）、推定値の分散は

$$\mathrm{Var}(\hat\beta_1)=\sum_{i=1}^{n}w_i^2\,\mathrm{Var}(u_i)$$

になる。第4回では、ここで仮定5（どの世帯も $\mathrm{Var}(u_i)=\sigma^2$）と $\sum w_i^2=1/SST$ を使って

$$\mathrm{Var}(\hat\beta_1)=\frac{\sigma^2}{SST}$$

を導いた。statsmodels が表示する通常の標準誤差は、この式の $\sigma^2$ を残差から推定し、平方根をとった値である。

### 6.1. 誤差の分散が世帯ごとに違う場合

今回のデータでは世帯 $i$ の誤差の分散が $\sigma_i^2=(0.1\times 所得_i)^2$ で、世帯ごとに違う。上の式の $\mathrm{Var}(u_i)$ に $\sigma_i^2$ を入れ、$w_i^2=(x_i-\bar x)^2/SST^2$ を使って書き直すと

$$\mathrm{Var}(\hat\beta_1)=\sum_{i=1}^{n}w_i^2\sigma_i^2=\frac{1}{SST}\sum_{i=1}^{n}a_i\sigma_i^2,\qquad a_i=\frac{(x_i-\bar x)^2}{SST}$$

となる。$a_i$ を200世帯で足すと1になるので、$\sum a_i\sigma_i^2$ は $a_i$ を重みとした $\sigma_i^2$ の平均（加重平均）である。仮定5が成り立てば、どの世帯の $\sigma_i^2$ も $\sigma^2$ なので、加重平均も $\sigma^2$ になり、第4回の式に戻る。

重み $a_i$ を百分率にして、所得との関係を図にする。

In [ ]:
a = (x - x.mean())**2 / sst
print(f'a_i の合計：{a.sum():.3f}')

fig, ax = plt.subplots()
ax.scatter(x, 100 * a, s=15)
ax.set(xlabel='所得（万円）', ylabel='重みの割合（%）')
plt.show()

$a_i$ の合計は1である。所得が平均（約52万円）に近い世帯の重みはほぼ0で、平均から離れるほど重みは大きくなる。所得の分布は高い側に長く伸びているので、特に重みの大きい世帯は高所得の側にある。所得が約128万円の世帯の重みは約9%で、200世帯の重みの平均（100% ÷ 200 ＝ 0.5%）の約18倍になる。

この高所得の世帯は、誤差の標準偏差も大きい。$\sigma_i^2$ の単純な平均と、$a_i$ を重みとした加重平均を比べよう。最後の行の `fit.scale` は、通常の標準誤差の計算に使う $\sigma^2$ の推定値（残差の二乗和を $n-2$ で割った値）である。

In [ ]:
print(f"σ_i² の単純な平均：{(df['sigma']**2).mean():.1f}")
print(f"σ_i² の加重平均：{(a * df['sigma']**2).sum():.1f}")
print(f"残差から推定した σ²：{fit.scale:.1f}")

加重平均は64.6で、単純な平均29.8の約2.2倍になった。重みの大きい高所得の世帯ほど誤差の分散も大きいからである。通常の標準誤差に使う推定値26.7は単純な平均に近く、世帯ごとの重みの違いを反映していない。

推定値の真の標準偏差は、上の分散の式の平方根である。`np.sqrt()` は平方根を求める。statsmodels の通常の標準誤差 `fit.bse['income']` と並べる。

In [ ]:
true_sd = np.sqrt((w**2 * df['sigma']**2).sum())
print(f"推定値の真の標準偏差：{true_sd:.4f}")
print(f"通常の標準誤差：{fit.bse['income']:.4f}")

通常の標準誤差は0.0201で、真の標準偏差0.0313の約6割しかない。標準誤差は信頼区間と $t$ 検定に使うので、小さく見積もると信頼区間は狭くなりすぎ、$t$ 値は大きくなりすぎる。

## 7. 頑健な標準誤差

実際のデータでは $\sigma_i^2$ が分からない。各世帯の観測は1つなので、世帯ごとに分散を推定することもできない。そこで、$\sigma_i^2$ の代わりに残差の二乗 $e_i^2$ を使う。

$$\widehat{\mathrm{Var}}(\hat\beta_1)=\sum_{i=1}^{n}w_i^2e_i^2$$

標本を何度も取り直すと、世帯 $i$ の $e_i^2$ の平均は $\sigma_i^2$ に近くなる。1つの標本の $e_i^2$ は $\sigma_i^2$ より大きいことも小さいこともあるが、多くの世帯について重みを付けて足すと、ずれが打ち消し合って $\sum w_i^2\sigma_i^2$ に近い値になる。この平方根を**不均一分散に頑健な標準誤差**（以下、頑健な標準誤差）という。誤差の分散が世帯ごとに違っても使えることを「頑健」と表している。

In [ ]:
robust_se = np.sqrt((w**2 * fit.resid**2).sum())
print(f"頑健な標準誤差：{robust_se:.4f}")

頑健な標準誤差は0.0284で、通常の標準誤差0.0201よりも真の標準偏差0.0313に近い。1つの標本の $e_i^2$ は偶然にばらつくので、真の値とぴったり同じにはならない。

statsmodels で `fit()` の括弧の中に `cov_type='HC0'` を書くと、標準誤差をこの式で計算する。HC は heteroskedasticity-consistent の略で、誤差の分散が観測ごとに違っても、観測数が多ければ正しい値に近づくことを表す。0 は、補正を加えない基本の形を指す。

In [ ]:
fit_hc0 = sm.OLS(df['consumption'], X).fit(cov_type='HC0')
print(f"所得の係数：{fit_hc0.params['income']:.3f}")
print(f"HC0 の標準誤差：{fit_hc0.bse['income']:.4f}")

標準誤差は上で計算した値と一致する。係数は通常の OLS と同じ0.296で、変わるのは標準誤差の計算方法だけである。

## 8. 標本を取り直したときの標準誤差

1つの標本では、標準誤差が偶然に真の値に近いこともある。第4回の第4節と同じように、所得を固定して誤差だけを1,000回作り直し、そのたびに傾きの推定値、通常の標準誤差、頑健な標準誤差を記録する。標準誤差が推定値のばらつきを正しく表していれば、その平均は推定値の標準偏差に近くなる。

下のセルでは、第3節で用意した `rng` を続けて使って誤差を作る。空のリスト `slopes`、`se_ordinary`、`se_robust` を用意し、`for` の繰り返しのたびに `.append()` で値を1つずつ加える。`range(1000)` は1,000回繰り返す指定で、何回目かの番号は使わないので、変数名を `_` にしている。

In [ ]:
slopes = []
se_ordinary = []
se_robust = []
for _ in range(1000):
    y_new = 10 + 0.3 * income + rng.normal(0, sigma)
    fit_new = sm.OLS(y_new, X).fit()
    slopes.append(fit_new.params['income'])
    se_ordinary.append(fit_new.bse['income'])
    se_robust.append(np.sqrt((w**2 * fit_new.resid**2).sum()))

3つのリストを表にまとめ、`.mean()` で平均を、`.std()` で標準偏差を求める。

In [ ]:
results = pd.DataFrame({'slope': slopes, 'se_ordinary': se_ordinary,
                        'se_robust': se_robust})
print(f"傾きの推定値の平均：{results['slope'].mean():.3f}")
print(f"傾きの推定値の標準偏差：{results['slope'].std():.4f}")
print(f"通常の標準誤差の平均：{results['se_ordinary'].mean():.4f}")
print(f"頑健な標準誤差の平均：{results['se_robust'].mean():.4f}")

傾きの推定値の平均は0.300で、真の値0.3に一致する。仮定5が成り立たなくても、仮定1〜4の下で OLS は不偏である（第4回の第3.2節）。推定値の標準偏差は0.0309で、第6.1節の式で計算した0.0313に近い。1,000回の結果にも乱数による変動があるので、ぴったりとは一致しない。

通常の標準誤差の平均0.0211は、推定値の標準偏差より約3割小さい。頑健な標準誤差の平均0.0296は、推定値の標準偏差に近い。

## 9. 住宅データの標準誤差

第2回の住宅データに戻る。47都道府県の総面積1km²当たりの人口密度（2024年、人/km²）と、持ち家住宅の1住宅当たりの延べ面積（2023年、m²）を使う。出典は総務省統計局[『統計でみる都道府県のすがた2026』](https://www.stat.go.jp/data/k-sugata/naiyou.html)である。

下のセルは第2回と同じ方法で、文字で書いた表を `StringIO()` で読み込める形にし、`pd.read_csv()` で表にする。

In [ ]:
from io import StringIO

housing_csv = """prefecture,density,floor_area
北海道,64.3,120.1
青森県,120.8,145.2
岩手県,75.0,145.8
宮城県,308.7,127.9
秋田県,77.1,154.3
山形県,108.4,159.9
福島県,126.4,139.2
茨城県,460.1,126.9
栃木県,294.2,128.7
群馬県,297.1,128.7
埼玉県,1930.6,106.4
千葉県,1212.3,109.0
東京都,6444.7,90.5
神奈川県,3817.4,97.9
新潟県,166.8,155.7
富山県,234.7,167.5
石川県,262.3,151.8
福井県,176.3,163.5
山梨県,177.1,132.6
長野県,146.5,144.1
岐阜県,180.4,141.9
静岡県,453.5,127.5
愛知県,1442.0,123.7
三重県,296.3,131.1
滋賀県,349.0,137.2
京都府,546.4,110.5
大阪府,4596.0,99.9
兵庫県,635.3,114.2
奈良県,348.1,127.8
和歌山県,186.3,123.8
鳥取県,151.4,147.2
島根県,95.7,151.1
岡山県,257.4,131.4
広島県,320.1,119.5
山口県,209.6,124.6
徳島県,165.2,133.8
香川県,488.6,132.1
愛媛県,224.8,123.7
高知県,92.4,117.3
福岡県,1020.9,114.8
佐賀県,322.9,138.9
長崎県,303.1,121.5
熊本県,229.0,125.0
大分県,171.1,123.5
宮崎県,133.6,115.2
鹿児島県,166.8,108.8
沖縄県,642.4,106.0"""
housing = pd.read_csv(StringIO(housing_csv))
housing.head()

`prefecture` は都道府県、`density` は人口密度、`floor_area` は持ち家の床面積である。説明変数は第2回と同じく人口密度の自然対数で、`housing['log_density'] = np.log(housing['density'])` によって、その列を表に加える。

In [ ]:
housing['log_density'] = np.log(housing['density'])
X_housing = sm.add_constant(housing[['log_density']])
fit_housing = sm.OLS(housing['floor_area'], X_housing).fit()
print(f"傾き：{fit_housing.params['log_density']:.2f}")
print(f"通常の標準誤差：{fit_housing.bse['log_density']:.2f}")

傾き −11.24 と標準誤差 1.85 は、第2回の値と同じである。頑健な標準誤差は、第7節と同じく `cov_type='HC0'` で計算する。

In [ ]:
fit_housing_hc0 = sm.OLS(housing['floor_area'], X_housing).fit(cov_type='HC0')
print(f"HC0 の標準誤差：{fit_housing_hc0.bse['log_density']:.2f}")

HC0 の標準誤差は1.46で、通常の標準誤差1.85より小さい。観測数が少ないときは、HC0 に補正を加えて少し大きくする方法も使われる。住宅データでその方法を使っても、標準誤差は通常の標準誤差より小さい。

### 9.1. 頑健な標準誤差が小さくなった理由

人工データでは頑健な標準誤差のほうが通常の標準誤差より大きかったが、住宅データでは小さい。第6.1節の式の所得を人口密度の対数に、$\sigma_i^2$ を $e_i^2$ に置き換えて理由を調べる。

下のセルでは、各県の重み $a_i$ を百分率にした列 `weight_pct` と、残差の二乗の列 `resid_sq` を表に加える。

In [ ]:
xh = housing['log_density']
housing['weight_pct'] = 100 * (xh - xh.mean())**2 / ((xh - xh.mean())**2).sum()
housing['resid_sq'] = fit_housing.resid**2

`.sort_values('weight_pct', ascending=False)` は表を `weight_pct` の大きい順に並べ替える。`ascending=False` が大きい順の指定で、付けなければ小さい順になる。`.head()` で上から5県を表示する。

In [ ]:
housing.sort_values('weight_pct', ascending=False).head().round(1)

重みが最も大きいのは東京都の18.3%で、大阪府、神奈川県、埼玉県、北海道が続く。重みの合計は100%なので47都道府県の平均は約2.1%で、東京都1つが平均の約9倍の重みをもつ。どれも人口密度が平均から大きく離れた都道府県で、北海道を除く4都府県の残差の二乗は20未満と小さい。

次に、残差の二乗の大きい順に並べる。

In [ ]:
housing.sort_values('resid_sq', ascending=False).head().round(1)

残差の二乗が最も大きいのは富山県の1256.0で、福井県、鹿児島県、北海道、高知県が続く。富山県、福井県、鹿児島県の重みは1%未満である。この3県の人口密度は平均に近いので、持ち家の床面積が回帰直線から大きく離れていても、頑健な標準誤差への影響は小さい。

残差の二乗の単純な平均と、`weight_pct` を100で割った値を重みとする加重平均を比べる。

In [ ]:
print(f"残差の二乗の単純な平均：{housing['resid_sq'].mean():.1f}")
print(f"残差の二乗の加重平均：{(housing['weight_pct'] / 100 * housing['resid_sq']).sum():.1f}")

加重平均108.3は単純な平均165.8の約65%である。通常の標準誤差は、第6.1節と同じく単純な平均に近い値を使う。重みの大きい県の残差が小さく、残差の大きい県の重みが小さいので、頑健な標準誤差は通常の標準誤差より小さくなった。

## 10. 2つの標準誤差の使い方

どちらの標準誤差が大きくなるかは、誤差の分散が大きい観測が、説明変数の平均から離れたところにあるか、平均に近いところにあるかで決まる。人工データのように平均から離れた観測で分散が大きければ、頑健な標準誤差のほうが大きい。住宅データのように平均に近い観測で残差が大きければ、頑健な標準誤差のほうが小さい。

実際のデータでは、誤差の分散が一定かどうかは分からない。誤差の分散が一定のときも、観測数が多ければ頑健な標準誤差は通常の標準誤差とほぼ同じ値になる。そのため、分散が一定だと考える理由がなければ、頑健な標準誤差を報告することが多い。

頑健な標準誤差で変わるのは標準誤差の計算方法で、係数は通常の OLS と同じである。第4回の欠落変数バイアスのように、仮定4が成り立たないために係数が真の値からずれている場合、そのずれは標準誤差の計算方法を変えても残る。

## 関数の資料

- [statsmodels：OLS の fit（`cov_type` の指定）](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.fit.html)
- [statsmodels：HC0 の標準誤差](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.RegressionResults.HC0_se.html)
- [NumPy：乱数の生成（Generator）](https://numpy.org/doc/stable/reference/random/generator.html)